In [1]:
import importlib
import sys
import os
import pickle

# performance imports for torch: torch kernel uses one core only.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch
import pandas as pd

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

In [2]:
# Event log and dataset configuration (must match the base loader)
event_log_location = '../../../../../../data/data/Sepsis.csv'
result_name = 'sepsis_all'
min_suffix_size = 5

event_log_properties = {
    'case_name': 'case:concept:name',
    'concept_name': 'concept:name',
    'timestamp_name': 'time:timestamp',
}

## Load artifacts from base loader and decision mining

In [3]:
# Load the Petri net discovered in the base loader
petri_net_path = '../../../../data/Sepsis/Petri_net/sepsis.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)
print('Petri net loaded from', petri_net_path)

# Load the normal (non-decision-labeled) tensor datasets from the base loader
train_set = torch.load('../../../../data/Sepsis/tensor_data/normal/sepsis_all_5_train.pkl', weights_only=False)
val_set   = torch.load('../../../../data/Sepsis/tensor_data/normal/sepsis_all_5_val.pkl', weights_only=False)
print(f'Train set: {len(train_set)} prefixes, Val set: {len(val_set)} prefixes')

# Load case IDs from saved CSVs
train_pref_df = pd.read_csv('../../../../data/Sepsis/raw_data/sepsis_all_5_train.csv')
val_pref_df   = pd.read_csv('../../../../data/Sepsis/raw_data/sepsis_all_5_val.csv')

unique_list_train = train_pref_df['case:concept:name'].dropna().unique().tolist()
unique_list_val   = val_pref_df['case:concept:name'].dropna().unique().tolist()
print(f'Unique train cases: {len(unique_list_train)}, val cases: {len(unique_list_val)}')

Petri net loaded from ../../../../data/Sepsis/Petri_net/sepsis.pkl
Train set: 9750 prefixes, Val set: 2297 prefixes
Unique train cases: 683, val cases: 157


## Decision-label the tensor-datasets

In [4]:
import data_processing.decision_labeling
importlib.reload(data_processing.decision_labeling)
from data_processing.decision_labeling import DecisionLabeler

from pm4py.objects.log.util import dataframe_utils
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignment_algo
from pm4py.utils import get_properties

# Decision mining artifact paths
decision_bundle_path = '../../../../data/Sepsis/Petri_net/data_aware_Petri_net/decision_places_bundle.json'
decision_model_dir   = '../../../../data/Sepsis/Petri_net/data_aware_Petri_net/models'

# Attributes used during decision mining (must match exactly)
dynamic_attributes = ['org:group',
                      'lifecycle:transition',
                      'case_elapsed_time',
                      'event_elapsed_time',
                      'Leucocytes',
                      'CRP',
                      'LacticAcid']

static_attributes = ['Age',
                     'InfectionSuspected',
                     'Diagnose',
                     'DiagnosticLacticAcid',
                     'DiagnosticBlood',
                     'DiagnosticArtAstrup',
                     'DiagnosticIC',
                     'DiagnosticSputum',
                     'DiagnosticLiquor',
                     'DiagnosticOther',
                     'DiagnosticUrinarySediment',
                     'DiagnosticECG',
                     'DiagnosticUrinaryCulture',
                     'DiagnosticXthorax',
                     'SIRSCritTachypnea',
                     'SIRSCritHeartRate',
                     'SIRSCriteria2OrMore',
                     'SIRSCritTemperature',
                     'SIRSCritLeucos',
                     'Hypotensie',
                     'Oligurie',
                     'Infusion',
                     'Hypoxie',
                     'DisfuncOrg']

print("Decision-labeling dynamic attributes:", dynamic_attributes)
print("Decision-labeling static attributes:", static_attributes)

Decision-labeling dynamic attributes: ['org:group', 'lifecycle:transition', 'case_elapsed_time', 'event_elapsed_time', 'Leucocytes', 'CRP', 'LacticAcid']
Decision-labeling static attributes: ['Age', 'InfectionSuspected', 'Diagnose', 'DiagnosticLacticAcid', 'DiagnosticBlood', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'DiagnosticUrinarySediment', 'DiagnosticECG', 'DiagnosticUrinaryCulture', 'DiagnosticXthorax', 'SIRSCritTachypnea', 'SIRSCritHeartRate', 'SIRSCriteria2OrMore', 'SIRSCritTemperature', 'SIRSCritLeucos', 'Hypotensie', 'Oligurie', 'Infusion', 'Hypoxie', 'DisfuncOrg']


In [5]:
# Load the original event log with timestamps and convert to pm4py column names
el_df = pd.read_csv(event_log_location)
el_df = el_df.rename(columns={
    event_log_properties['case_name']:      'case:concept:name',
    event_log_properties['concept_name']:   'concept:name',
    event_log_properties['timestamp_name']: 'time:timestamp',
})
el_df['time:timestamp'] = pd.to_datetime(el_df['time:timestamp'])
el_df = dataframe_utils.convert_timestamp_columns_in_df(el_df)
print(f"Event log: {len(el_df)} events, {el_df['case:concept:name'].nunique()} cases")

Event log: 15214 events, 1049 cases


In [6]:
# Compute optimal alignments for all case IDs across train / val
all_case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))

# Filter event log to relevant cases
el_aligned = el_df[el_df['case:concept:name'].isin(all_case_ids)].copy()

params = get_properties(el_aligned)
params["ret_tuple_as_trans_desc"] = True

aligned_results = alignment_algo.apply(el_aligned, net, im, fm, parameters=params)
all_alignments = [r['alignment'] for r in aligned_results]

# Case-ID ordering must match the alignment list (pm4py preserves DataFrame group order)
sorted_case_ids = (el_aligned.drop_duplicates(subset='case:concept:name', keep='first')['case:concept:name'].tolist())
print(f"Computed {len(all_alignments)} alignments for {len(sorted_case_ids)} cases")

aligning log, completed variants ::   0%|          | 0/692 [00:00<?, ?it/s]

Computed 840 alignments for 840 cases


In [7]:
# Create decision labeler (replaces el_loader.label_dataset)
labeler = DecisionLabeler(petri_net=(net, im, fm),
                          decision_model_dir=decision_model_dir,
                          decision_places_bundle_path=decision_bundle_path,
                          dynamic_attributes=dynamic_attributes,
                          static_attributes=static_attributes)

# Decision-label train set (offline, from optimal alignments)
labeler.label_dataset_offline(train_set, el_aligned, sorted_case_ids, all_alignments)
print(f"Train set labeled: {len(train_set)} prefixes")

/home/PSPLab/ProbabilisticSuffixPredictionLab/decision_aware_augmentation_for_PPM/src/notebooks/suffix_prediction/data_loader/../../../decision_mining/function_estimator_catboost_advanced.py:747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_raw[col] = np.nan
/home/PSPLab/ProbabilisticSuffixPredictionLab/decision_aware_augmentation_for_PPM/src/notebooks/suffix_prediction/data_loader/../../../decision_mining/function_estimator_catboost_advanced.py:747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_raw[col] = np.nan
/home/PSPL

Train set labeled: 9750 prefixes


In [8]:
# Decision-label val set (offline)
labeler.label_dataset_offline(val_set, el_aligned, sorted_case_ids, all_alignments)
print(f"Val set labeled: {len(val_set)} prefixes")

/home/PSPLab/ProbabilisticSuffixPredictionLab/decision_aware_augmentation_for_PPM/src/notebooks/suffix_prediction/data_loader/../../../decision_mining/function_estimator_catboost_advanced.py:747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_raw[col] = np.nan
/home/PSPLab/ProbabilisticSuffixPredictionLab/decision_aware_augmentation_for_PPM/src/notebooks/suffix_prediction/data_loader/../../../decision_mining/function_estimator_catboost_advanced.py:747: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_raw[col] = np.nan
/home/PSPL

Val set labeled: 2297 prefixes


In [9]:
# Identify the concept:name feature index for guard tensors
activity_feature_name = 'concept:name'
concept_name_feature_idx = next(i for i, cat in enumerate(train_set.all_categories[0]) if cat[0] == activity_feature_name)

# Convert sparse decision_data → dense guard tensors so they are stored in the .pkl
for name, ds in [("Train", train_set), ("Val", val_set)]:
    ds.prepare_guard_tensors(concept_name_feature_idx)
    
    # guard_targets is a tensor of shape [N, T, C] on the dataset, or [B, S, C] after batching/slicing. It stores a soft probability distribution z_i over next-event label classes for each event position.
    # guard_mask is a tensor of shape [N, T] or [B, S]. It is just an indicator: 1 where that event position has a usable decision label, 0 where it does not.
    # guard_confidence is a tensor of shape [N, T] on the dataset. It stores a scalar confidence c_i for each labeled event position, usually the maximum probability in the decision distribution if no explicit confidence was stored.
    
    print(f"{name}: guard_targets {ds._guard_targets.shape}, "
          f"guard_mask {ds._guard_mask.shape}, "
          f"guard_confidence {ds._guard_confidence.shape}")

Train: guard_targets torch.Size([9750, 50, 18]), guard_mask torch.Size([9750, 50]), guard_confidence torch.Size([9750, 50])
Val: guard_targets torch.Size([2297, 50, 18]), guard_mask torch.Size([2297, 50]), guard_confidence torch.Size([2297, 50])


In [10]:
# Save decision-labeled datasets
dl_dir = '../../../../data/Sepsis/tensor_data/decision_labeled/'
os.makedirs(dl_dir, exist_ok=True)

sfx = '_' + str(train_set.min_suffix_size)
torch.save(train_set, dl_dir + result_name + sfx + '_train.pkl')
torch.save(val_set,   dl_dir + result_name + sfx + '_val.pkl')
print("Saved decision-labeled datasets to", dl_dir)

Saved decision-labeled datasets to ../../../../data/Sepsis/tensor_data/decision_labeled/


In [11]:
# Verify: inspect a sample of decision labels
for name, ds in [("Train", train_set), ("Val", val_set)]:
    dd = ds.decision_data
    n = len(dd) if not isinstance(dd, torch.Tensor) else dd.size(0)
    print(f"\n{name}: {n} prefixes")
    if isinstance(dd, list) and n > 0:
        sample = dd[0]
        print(f"  First prefix decisions ({len(sample)} events):")
        for j, (place, dist, c_i) in enumerate(sample[:5]):
            print(f"e_{j}: place={place}, z_i={dist}, c_i={c_i}")


Train: 9750 prefixes
  First prefix decisions (6 events):
e_0: place=⊥, z_i={}, c_i=0.0
e_1: place=p_22, z_i={'Admission IC': 0.001457627330320267, 'Admission NC': 0.015557026493287718, 'CRP': 0.28741560359019286, 'EOS': 0.0010435653404780863, 'ER Registration': 0.00203579941069827, 'ER Sepsis Triage': 0.0035432312638088975, 'ER Triage': 0.001999144979619084, 'IV Antibiotics': 0.006610813773405396, 'IV Liquid': 0.008850304776778879, 'LacticAcid': 0.666989580131262, 'Leucocytes': 0.002643815781675836, 'Release A': 0.0011952739791551783, 'Release B': 0.00019148082726736917, 'Release C': 9.549700355160406e-05, 'Release D': 9.152045479772629e-05, 'Release E': 0.00011933495250549164, 'Return ER': 0.00016037991119533725}, c_i=0.666989580131262
e_2: place=p_27, z_i={'Admission IC': 0.0025247694982448526, 'Admission NC': 0.017575938234796817, 'CRP': 0.0016832590633163124, 'EOS': 0.0025455764996513306, 'ER Registration': 0.0009001212393312738, 'ER Sepsis Triage': 0.01589734171880993, 'ER Triag

In [12]:
# Example output of one train sample
print(train_set.all_categories[0][0][2])
print(train_set.categorical_tensors[0][0])
print(train_set.decision_data[0])

{'Admission IC': 1, 'Admission NC': 2, 'CRP': 3, 'EOS': 4, 'ER Registration': 5, 'ER Sepsis Triage': 6, 'ER Triage': 7, 'IV Antibiotics': 8, 'IV Liquid': 9, 'LacticAcid': 10, 'Leucocytes': 11, 'Release A': 12, 'Release B': 13, 'Release C': 14, 'Release D': 15, 'Release E': 16, 'Return ER': 17}
tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  5, 11,  3, 10,  7,  6])
[('⊥', {}, 0.0), ('p_22', {'Admission IC': 0.001457627330320267, 'Admission NC': 0.015557026493287718, 'CRP': 0.28741560359019286, 'EOS': 0.0010435653404780863, 'ER Registration': 0.00203579941069827, 'ER Sepsis Triage': 0.0035432312638088975, 'ER Triage': 0.001999144979619084, 'IV Antibiotics': 0.006610813773405396, 'IV Liquid': 0.008850304776778879, 'LacticAcid': 0.666989580131262, 'Leucocytes': 0.002643815781675836, 'Release A': 0.0011952739791551783, 'Release B': 0.000191